In [ ]:
#clean food data set
import pandas as pd
df = pd.read_csv("StateAndCountyData.csv", header=None,
                 names=["FIPS", "State", "County", "Variable", "Value"],
                 dtype={"FIPS": str})
df_ca = df[df["State"] == "CA"]
variables = ["PCT_LACCESS_POP19", "PCT_LACCESS_LOWI19", "PCT_LACCESS_HHNV19",
             "GROCPTH20", "SUPERCPTH20", "SNAPSPTH23",
             "CONVSPTH20", "FFRPTH20", "MEDHHINC21"]
df_ca_filtered = df_ca[df_ca["Variable"].isin(variables)]
df_food = df_ca_filtered.pivot(index=["FIPS", "County"],
                                columns="Variable",
                                values="Value").reset_index()
df_food.columns.name = None
df_food[variables] = df_food[variables].astype(float)
df_food = df_food.sort_values("County").reset_index(drop=True)
df_food

/tmp/ipykernel_514/1592960392.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("StateAndCountyData.csv", header=None,


,FIPS,County,CONVSPTH20,FFRPTH20,GROCPTH20,MEDHHINC21,PCT_LACCESS_HHNV19,PCT_LACCESS_LOWI19,PCT_LACCESS_POP19,SNAPSPTH23,SUPERCPTH20
0,06001,Alameda,0.197314,0.827156,0.219572,108971.0,0.342808,1.117399,8.083675,0.545731,0.016242
1,06003,Alpine,-9999.000000,2.680965,-9999.000000,87570.0,6.022490,16.785139,56.969242,-9999.000000,-9999.000000
2,06005,Amador,0.324327,0.573809,0.299379,68159.0,3.163143,0.095632,0.174187,0.763980,-9999.000000
3,06007,Butte,0.329034,0.737976,0.188019,62982.0,1.979342,6.450867,19.113434,0.926945,0.018802
4,06009,Calaveras,0.410296,0.604647,0.259135,68298.0,1.547478,5.989503,14.736117,0.985243,-9999.000000
5,06011,Colusa,0.556638,0.417478,0.324705,60725.0,2.258987,12.356255,39.329746,1.283756,-9999.000000
6,06013,Contra Costa,0.200463,0.678623,0.194388,110595.0,0.766959,2.867471,23.228312,0.542727,0.013885
7,06015,Del Norte,0.178776,0.321796,0.143021,48108.0,3.967303,14.445117,34.407192,0.983177,-9999.000000
8,06017,El Dorado,0.285085,0.673837,0.171051,87689.0,1.906889,4.258567,25.222891,0.636368,0.020733
9,06019,Fresno,0.298726,0.731329,0.252768,63140.0,0.889459,5.349311,13.327069,0.916412,0.019982


In [ ]:
#clean health data set
file = "CountyHealthRankings.xlsx"
df_rank = pd.read_excel(
    file,
    sheet_name="Ranked Measure Data",
    header=1
)
df_add = pd.read_excel(
    file,
    sheet_name="Additional Measure Data",
    header=1
)
df_obesity = df_rank[["FIPS", "County", "% Adults with Obesity"]]
df_diabetes = df_add[["FIPS", "County", "% Adults with Diabetes"]]
df_obesity["FIPS"] = df_obesity["FIPS"].astype(str).str.zfill(5)
df_diabetes["FIPS"] = df_diabetes["FIPS"].astype(str).str.zfill(5)
df_health = pd.merge(
    df_obesity,
    df_diabetes[["FIPS", "% Adults with Diabetes"]],
    on="FIPS",
    how="inner")
df_health

/tmp/ipykernel_514/538965661.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_obesity["FIPS"] = df_obesity["FIPS"].astype(str).str.zfill(5)
/tmp/ipykernel_514/538965661.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_diabetes["FIPS"] = df_diabetes["FIPS"].astype(str).str.zfill(5)


,FIPS,County,% Adults with Obesity,% Adults with Diabetes
0,06000,NaN,26.2,9.4
1,06001,Alameda,23.5,9.1
2,06003,Alpine,29.9,9.9
3,06005,Amador,29.4,9.0
4,06007,Butte,31.7,9.5
5,06009,Calaveras,30.1,8.7
6,06011,Colusa,32.3,11.8
7,06013,Contra Costa,23.9,9.4
8,06015,Del Norte,33.4,10.9
9,06017,El Dorado,26.0,7.9


In [ ]:
df_cancer = pd.read_csv("cali_cancer_rates.csv", skiprows=8)
df_cancer = df_cancer.iloc[2:60]
df_cancer = df_cancer[[
    "County",
    "FIPS",
    "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000"]]
df_cancer["County"] = (
    df_cancer["County"]
    .str.replace(" County", "", regex=False)   # remove " County"
    .str.replace(r"\(.*\)", "", regex=True)    # remove "(7)" etc
    .str.strip())
df_cancer = df_cancer.rename(columns={
    "Age-Adjusted Incidence Rate([rate note]) - cases per 100,000": "Cancer Incidence Rate"})
df_cancer["Cancer Incidence Rate"] = pd.to_numeric(df_cancer["Cancer Incidence Rate"], errors="coerce")
df_cancer

,County,FIPS,Cancer Incidence Rate
2,Butte,6007.0,42.5
3,San Benito,6069.0,40.3
4,Colusa,6011.0,40.1
5,Tehama,6103.0,39.8
6,Lake,6033.0,39.6
7,Merced,6047.0,39.4
8,Amador,6005.0,39.3
9,Yuba,6115.0,38.9
10,Calaveras,6009.0,38.4
11,Shasta,6089.0,37.9


In [ ]:
#merge
import numpy as np
df_final = pd.merge(df_food, df_health, on=["FIPS", "County"])
df_final = df_final.rename(columns={
    "PCT_LACCESS_POP19": "% Population with low access to store",
    "PCT_LACCESS_LOWI19": "% Population low income & low access to store",
    "PCT_LACCESS_HHNV19": "% Households, no car & low access to store",
    "GROCPTH20": "Grocery stores/1,000 population",
    "SUPERCPTH20": "Supercenters/1,000 population",
    "SNAPSPTH23": "SNAP-authorized stores/1,000 population",
    "FOODINSEC_21_23": "% Household food insecurity",
    "CONVSPTH20": "Convenience stores/1,000 population",
    "FFRPTH20": "Fast-food restaurants/1,000 population",
    "MEDHHINC21": "Median household income"
})
num_cols = df_final.select_dtypes(include='number')
df_final[num_cols.columns] = num_cols.mask(num_cols < 0)
df_final["Retail Food Environment Index"] = (
    (df_final["Fast-food restaurants/1,000 population"] +
     df_final["Convenience stores/1,000 population"])
    /
    (df_final["Grocery stores/1,000 population"] +
     df_final["Supercenters/1,000 population"])
)
df_final["FIPS"] = df_final["FIPS"].astype(int)
df_cancer["FIPS"] = df_cancer["FIPS"].astype(int)
df_final = pd.merge(df_final, df_cancer, on=["FIPS", "County"], how="left")
df_final

,FIPS,County,"Convenience stores/1,000 population","Fast-food restaurants/1,000 population","Grocery stores/1,000 population",Median household income,"% Households, no car & low access to store",% Population low income & low access to store,% Population with low access to store,"SNAP-authorized stores/1,000 population","Supercenters/1,000 population",% Adults with Obesity,% Adults with Diabetes,Retail Food Environment Index,Cancer Incidence Rate
0,6001,Alameda,0.197314,0.827156,0.219572,108971.0,0.342808,1.117399,8.083675,0.545731,0.016242,23.5,9.1,4.344388,31.7
1,6003,Alpine,NaN,2.680965,NaN,87570.0,6.022490,16.785139,56.969242,NaN,NaN,29.9,9.9,NaN,NaN
2,6005,Amador,0.324327,0.573809,0.299379,68159.0,3.163143,0.095632,0.174187,0.763980,NaN,29.4,9.0,NaN,39.3
3,6007,Butte,0.329034,0.737976,0.188019,62982.0,1.979342,6.450867,19.113434,0.926945,0.018802,31.7,9.5,5.159091,42.5
4,6009,Calaveras,0.410296,0.604647,0.259135,68298.0,1.547478,5.989503,14.736117,0.985243,NaN,30.1,8.7,NaN,38.4
5,6011,Colusa,0.556638,0.417478,0.324705,60725.0,2.258987,12.356255,39.329746,1.283756,NaN,32.3,11.8,NaN,40.1
6,6013,Contra Costa,0.200463,0.678623,0.194388,110595.0,0.766959,2.867471,23.228312,0.542727,0.013885,23.9,9.4,4.220833,34.2
7,6015,Del Norte,0.178776,0.321796,0.143021,48108.0,3.967303,14.445117,34.407192,0.983177,NaN,33.4,10.9,NaN,24.5
8,6017,El Dorado,0.285085,0.673837,0.171051,87689.0,1.906889,4.258567,25.222891,0.636368,0.020733,26.0,7.9,5.000000,35.6
9,6019,Fresno,0.298726,0.731329,0.252768,63140.0,0.889459,5.349311,13.327069,0.916412,0.019982,37.6,13.5,3.776557,33.1


In [ ]:
df_final.to_csv('datasci112_final.csv', index=False)
from google.colab import files
files.download('datasci112_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>